# Import

In [ ]:
!pip install transformers datasets accelerate --quiet

from transformers import (
    GPT2Tokenizer,
    GPT2LMHeadModel,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
    pipeline,
    set_seed
)

from datasets import load_dataset, Dataset
import torch
import math
import re

set_seed(42)

## Dataset

In [ ]:
raw = load_dataset("roneneldan/TinyStories", split="train[:8000]")

# Menentukan genre berdasarkan kata kunci
def has_word(t, w):
    return re.search(rf"\b{re.escape(w)}\b", t.lower()) is not None

def get_genre(t):
    if any(has_word(t, w) for w in [
        "dragon", "magic", "fairy", "fairies",
        "wizard", "princess", "castle"
    ]):
        return "fantasy"

    if any(has_word(t, w) for w in [
        "adventure", "explore", "forest",
        "journey", "treasure", "map"
    ]):
        return "adventure"

    if any(has_word(t, w) for w in [
        "friend", "friends", "friendship",
        "share", "help", "together", "kind"
    ]):
        return "friendship"

    return None

# Menentukan tema berdasarkan kata kunci
def get_theme(t):
    t = t.lower()

    for w in [
        "dragon", "princess", "robot", "dog", "cat",
        "rabbit", "bird", "treasure", "ocean", "forest"
    ]:
        if has_word(t, w):
            return w

    return "everyday life"

# Memperkirakan kelompok usia berdasarkan panjang cerita
def get_age(n_words):
    if n_words < 110:
        return 5
    elif n_words < 150:
        return 7
    return 9

# Menentukan gaya cerita berdasarkan usia
def get_style(age):
    if age <= 5:
        return "use simple words and short sentences"
    elif age <= 7:
        return "use clear language and a simple story plot"

    return "use richer descriptions and a more detailed plot"

# Memproses dataset
texts = []
count = {"fantasy": 0, "adventure": 0, "friendship": 0}

for s in raw["text"]:
    s = s.strip()
    n = len(s.split())

    # Mengabaikan cerita yang terlalu pendek atau panjang
    if not 60 <= n <= 200:
        continue

    genre = get_genre(s)

    if genre is None:
        continue

    # Membatasi jumlah cerita setiap genre
    if count[genre] >= 600:
        continue

    count[genre] += 1

    age = get_age(n)
    theme = get_theme(s)
    style = get_style(age)

    texts.append(
        f"Age: {age} | Genre: {genre} | Theme: {theme}\n"
        f"Style: {style}\n"
        f"Story: {s}"
    )


print("Jumlah cerita per genre:", count)

dataset = Dataset.from_dict({"text": texts}).train_test_split(
    test_size=0.05,
    seed=42
)

print(dataset)
print("\nContoh data:")
print(dataset["train"][0]["text"][:400])

Jumlah cerita per genre: {'fantasy': 251, 'adventure': 600, 'friendship': 600}
DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 1378
    })
    test: Dataset({
        features: ['text'],
        num_rows: 73
    })
})

Contoh data:
Age: 9 | Genre: friendship | Theme: everyday life
Style: use richer descriptions and a more detailed plot
Story: Once upon a time, there was a pirate who lived in a far away land. He was a mean pirate and was very intelligent. He looked for ways to steal from others.

One day, he noticed two young kids playing nearby. The pirate decided to steal their toys. He waited until the kids were not lookin


## Tokenisasi & Fine-Tuning GPT-2

In [ ]:
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

# Mengubah teks menjadi token
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=320
    )


tokenized_datasets = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

# Memuat GPT-2
model = GPT2LMHeadModel.from_pretrained("gpt2")

# Konfigurasi fine-tuning
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    learning_rate=5e-5,
    save_strategy="no",
    logging_steps=50,
    report_to="none",
    fp16=torch.cuda.is_available()
)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    data_collator=data_collator
)

# Evaluasi sebelum fine-tuning
loss_before = trainer.evaluate()["eval_loss"]

# Melatih model
trainer.train()

# Evaluasi setelah fine-tuning
loss_after = trainer.evaluate()["eval_loss"]


print(f"Perplexity sebelum fine-tuning: {math.exp(loss_before):.2f}")
print(f"Perplexity sesudah fine-tuning: {math.exp(loss_after):.2f}")

Map:   0%|          | 0/1378 [00:00<?, ? examples/s]

Map:   0%|          | 0/73 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Training Loss,Validation Loss,Step
No log,2.759464,0


Step,Training Loss
50,2.144178
100,1.891399
150,1.815657
200,1.727880
250,1.704520
300,1.683852
350,1.698676
400,1.614040
450,1.626292
500,1.626829


Training Loss,Validation Loss,Step
1.626829,1.646772,519


Perplexity sebelum fine-tuning: 15.79
Perplexity sesudah fine-tuning: 5.19


## Text Generation

In [ ]:
model.eval()
device = 0 if torch.cuda.is_available() else -1

# Model GPT-2 setelah fine-tuning
tuned = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device=device
)

# Model GPT-2 asli sebagai pembanding
base = pipeline(
    "text-generation",
    model="gpt2",
    tokenizer=tokenizer,
    device=device
)

# Membuat personalized prompt
def build_prompt(age, genre, theme, opening=""):
    style = get_style(age)

    return (
        f"Age: {age} | Genre: {genre} | Theme: {theme}\n"
        f"Style: {style}\n"
        f"Story: {opening}"
    ).rstrip()

# Menghasilkan cerita
def make_story(generator, prompt, n=1, max_new_tokens=140, **params):
    outputs = generator(
        prompt,
        max_new_tokens=max_new_tokens,
        num_return_sequences=n,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
        **params
    )

    opening = prompt.split("Story:", 1)[1].strip()
    stories = []

    for o in outputs:
        t = o["generated_text"].split("Story:", 1)[1].strip()
        # Memotong teks pada akhir kalimat jika tersedia
        cut = max(
            t.rfind("."),
            t.rfind("!"),
            t.rfind("?")
        )
        if cut >= len(opening):
            t = t[:cut + 1]

        stories.append(t)

    return stories

# Prompt yang sama untuk membandingkan kedua model
prompt = build_prompt(
    7,
    "fantasy",
    "dragon",
    "Once upon a time, there was a little dragon who wanted to be a hero."
)


for name, generator in [
    ("GPT-2 ASLI", base),
    ("GPT-2 FINE-TUNED", tuned)
]:
    set_seed(42)

    story = make_story(
        generator,
        prompt,
        max_new_tokens=140,
        temperature=0.8,
        top_p=0.9
    )[0]

    print(f"\n===== {name} =====")
    print("PROMPT:")
    print(prompt)
    print("\nCERITA:")
    print(story)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]


===== GPT-2 ASLI =====
PROMPT:
Age: 7 | Genre: fantasy | Theme: dragon
Style: use clear language and a simple story plot
Story: Once upon a time, there was a little dragon who wanted to be a hero.

CERITA:
Once upon a time, there was a little dragon who wanted to be a hero. He was a mage who lived in a palace. He would lead a people called the Dragon Lords. The dragons would fight for him, and the dragon would be his heir.
A few years ago, he met a woman who wanted to become the Dragon Lord. She was beautiful and had a child, a girl named Lola. She wanted to become the Dragon Lord, but she could not have a child.
The Dragon Lords would rule over a kingdom of dragons and they would try to make sure that she was not a child.
A few days ago, the Dragon Lords came and made Lola a dragon. They had made a pact with Lola.

===== GPT-2 FINE-TUNED =====
PROMPT:
Age: 7 | Genre: fantasy | Theme: dragon
Style: use clear language and a simple story plot
Story: Once upon a time, there was a little 

## Personalized Prompting

In [ ]:
profiles = [
    (5, "fantasy", "dragon",
     "Once upon a time, there was a little dragon who wanted to be a hero."),

    (9, "fantasy", "dragon",
     "Once upon a time, there was a dragon who wanted to be a hero."),

    (7, "adventure", "treasure",
     "One day, a brave girl found an old map in the forest."),

    (7, "friendship", "dog",
     "Max the dog was sad because he had no friends."),

    (5, "friendship", "rabbit",
     "A little rabbit wanted to share his carrots.")
]


for i, (age, genre, theme, opening) in enumerate(profiles, 1):

    prompt = build_prompt(age, genre, theme, opening)

    # Panjang generasi dibedakan berdasarkan usia
    max_tokens = 100 if age <= 5 else 140 if age <= 7 else 170

    set_seed(42)

    story = make_story(
        tuned,
        prompt,
        max_new_tokens=max_tokens,
        temperature=0.8,
        top_p=0.9
    )[0]

    print(f"\n========== CONTOH {i} ==========")
    print("PROMPT:")
    print(prompt)

    print("\nCERITA:")
    print(story)


========== CONTOH 1 ==========
PROMPT:
Age: 5 | Genre: fantasy | Theme: dragon
Style: use simple words and short sentences
Story: Once upon a time, there was a little dragon who wanted to be a hero.

CERITA:
Once upon a time, there was a little dragon who wanted to be a hero. He was so excited to be a hero and had so much fun. But he didn't have a dragon costume!

One day, the dragon's parents wanted him to go on an adventure. The dragon felt sad because he didn't have a costume and wanted to show everyone. So, the dragon decided to go on a magical journey. He ran and jumped, but he couldn't make it past the gates. He was sad because he was sad because he had never seen a dragon before.

========== CONTOH 2 ==========
PROMPT:
Age: 9 | Genre: fantasy | Theme: dragon
Style: use richer descriptions and a more detailed plot
Story: Once upon a time, there was a dragon who wanted to be a hero.

CERITA:
Once upon a time, there was a dragon who wanted to be a hero. He was so excited to be a h

## Top-K vs Top-p (Nucleus) Sampling

In [ ]:
opening = (
    "Once upon a time, there was a little dragon "
    "who wanted to be a hero."
)

prompt = build_prompt(
    7,
    "fantasy",
    "dragon",
    opening
)


# Menghitung keragaman bigram
def distinct_2(stories, opening=""):
    bigrams = []

    for s in stories:
        # Menghilangkan pembuka yang sama
        if opening and s.startswith(opening):
            s = s[len(opening):]
        # Memisahkan kata dan mengabaikan tanda baca
        words = re.findall(r"\b\w+\b", s.lower())

        bigrams.extend(zip(words, words[1:]))

    return len(set(bigrams)) / max(1, len(bigrams))


# Konfigurasi sampling
configs = {
    "Top-K k=10": dict(
        top_k=10,
        top_p=1.0,
        temperature=1.0
    ),

    "Top-K k=50": dict(
        top_k=50,
        top_p=1.0,
        temperature=1.0
    ),

    "Top-p p=0.7": dict(
        top_k=0,
        top_p=0.7,
        temperature=1.0
    ),

    "Top-p p=0.9": dict(
        top_k=0,
        top_p=0.9,
        temperature=1.0
    ),

    "Top-p p=0.95": dict(
        top_k=0,
        top_p=0.95,
        temperature=1.0
    )
}

# Menghasilkan lima cerita untuk setiap konfigurasi
samples = {}

print(f"{'Strategi':<16}{'Distinct-2':>12}{'Rata-rata kata':>17}")

for name, cfg in configs.items():

    set_seed(42)

    samples[name] = make_story(
        tuned,
        prompt,
        n=5,
        **cfg
    )

    d2 = distinct_2(samples[name], opening)

    avg_words = (
        sum(len(s.split()) for s in samples[name])
        / max(1, len(samples[name]))
    )

    print(f"{name:<16}{d2:>12.3f}{avg_words:>17.1f}")

# Menampilkan satu contoh dari setiap konfigurasi
for name, stories in samples.items():

    print(f"\n========== {name} ==========")
    print(stories[0])

Strategi          Distinct-2   Rata-rata kata
Top-K k=10             0.756            119.4
Top-K k=50             0.808            124.4
Top-p p=0.7            0.694            120.4
Top-p p=0.9            0.816            109.2
Top-p p=0.95           0.810            122.6

========== Top-K k=10 ==========
Once upon a time, there was a little dragon who wanted to be a hero. He was very brave and had lots of courage.

When the sun went down, a little girl named Sarah wanted to be a hero. She wanted to go on an adventure.

Sarah's mommy asked Sarah if she could help a little girl, so Sarah did, and the little girl was so happy.

Sarah and her little girl went back home and played together. They played until they ran out of energy. Soon, there was nothing left for Sarah to do anymore.

Sarah's mommy told her that they could go back to their castle and play some games, but that was too hard to do.

========== Top-K k=50 ==========
Once upon a time, there was a little dragon who wanted to